The New Notebook: This table will look at the hourly weather and output a mathematically derived surge_multiplier (e.g., 1.5x delivery fee) and a road_safety_status.

Here is the PySpark code to generate this highly practical business table:

In [0]:
from pyspark.sql import functions as F

# 1. Read the highly granular Silver hourly data
df_silver = spark.read.table("weather.silver.silver_weather")

# 2. Build the Surge Pricing and Safety Logic
df_gold_logistics = df_silver.select(
    "date",
    "hour",
    "temperature_2m",
    "rain",
    "snowfall",
    "wind_speed_10m"
).withColumn(
    # Create the Base Multiplier
    "base_multiplier", F.lit(1.0)
).withColumn(
    # Add to the multiplier based on rain severity
    "rain_surge",
    F.when(F.col("rain") > 10, 0.8)  # Heavy rain: +80% fee
     .when(F.col("rain") > 2, 0.3)   # Light rain: +30% fee
     .otherwise(0.0)
).withColumn(
    # Add to the multiplier based on snow (very dangerous in hills)
    "snow_surge",
    F.when(F.col("snowfall") > 2, 1.5) # Heavy snow: +150% fee
     .when(F.col("snowfall") > 0, 0.5) # Light snow: +50% fee
     .otherwise(0.0)
).withColumn(
    # Calculate the Final Surge Pricing Multiplier
    "final_surge_multiplier",
    F.round(F.col("base_multiplier") + F.col("rain_surge") + F.col("snow_surge"), 2)
).withColumn(
    # Generate an automated safety dispatch warning for the operations team
    "dispatch_safety_warning",
    F.when(F.col("snowfall") > 5, "CRITICAL: Suspend 2-Wheeler Deliveries")
     .when(F.col("wind_speed_10m") > 40, "WARNING: High Wind Risk for Bikes")
     .when(F.col("rain") > 10, "WARNING: Low Visibility & Slippery Roads")
     .otherwise("CLEAR: Safe Operating Conditions")
)

# 3. Clean up the intermediate calculation columns to keep the final table neat
df_gold_logistics_final = df_gold_logistics.drop("base_multiplier", "rain_surge", "snow_surge")

# Sort by most recent dates to see the latest operational risks
df_gold_logistics_final = df_gold_logistics_final.orderBy(F.col("date").desc(), F.col("hour").desc())

# 4. Save the table
(df_gold_logistics_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather.gold.gold_logistics_surge_pricing"))

print("🛵 Dynamic Surge Pricing & Logistics Risk table successfully built!")